# Customer Health Score 3.0 - End-to-End Notebook
This notebook runs the full pipeline in one place, from data adaptation/generation to Power BI export.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().parents[0]
if str(root) not in sys.path:
    sys.path.append(str(root))

print(f'Project root: {root}')

## 1) Choose data source
Run either the TWCS adapter (public dataset) or synthetic generation.

In [ ]:
from src.twitter_kaggle_adapter import adapt_twitter_dataset
from src.data_generation import generate_data

twcs_path = root.parent / 'DS' / 'twcs' / 'twcs.csv'
use_public_dataset = twcs_path.exists()

if use_public_dataset:
    print(f'Using public TWCS dataset at: {twcs_path}')
    adapt_twitter_dataset(source_path=twcs_path, max_rows=120000, seed=42)
else:
    print('TWCS not found. Falling back to synthetic data generation.')
    generate_data()

## 2) Run full business pipeline
This includes: preprocessing, sentiment, features, modeling, health score, expected value, action policy, and Power BI export.

In [ ]:
from src.run_pipeline import run_pipeline

run_pipeline(generate=False)

## 3) Quick sanity checks

In [ ]:
import pandas as pd

processed = root / 'data' / 'processed'
model_metrics = pd.read_csv(processed / 'model_metrics.csv')
health_scores = pd.read_csv(processed / 'health_scores.csv')
customer_actions = pd.read_csv(processed / 'customer_actions.csv')
campaign_scenarios = pd.read_csv(processed / 'campaign_scenarios.csv')

display(model_metrics)
display(health_scores['health_segment'].value_counts().rename_axis('segment').reset_index(name='customers'))
display(customer_actions[['customer_id', 'next_best_action', 'priority_score', 'expected_net_value']].head(10))
display(campaign_scenarios)

## 4) Output artifacts
The main Power BI input is `data/processed/power_bi_export.xlsx`.

In [ ]:
from pathlib import Path

output_file = root / 'data' / 'processed' / 'power_bi_export.xlsx'
print(f'Power BI export exists: {output_file.exists()}')
print(output_file)